In [15]:
import pandas as pd

In [16]:
df = pd.read_csv("../data/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


Data preparation

In [ ]:
num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(0.0)

cat_cols = df.select_dtypes(include=["str"]).columns
df[cat_cols] = df[cat_cols].fillna("NA")

df.head(10)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
5,referral,manufacturing,employed,africa,43841.0,3,5,0.77,1
6,NA,education,employed,north_america,58738.0,3,9,0.78,1
7,paid_ads,technology,employed,north_america,95059.0,2,4,0.41,1
8,organic_search,healthcare,employed,asia,62565.0,3,6,0.52,1
9,referral,retail,student,asia,12000.0,2,2,0.35,0


Question 1:

In [28]:
df.industry.value_counts(), df.industry.mode()

(industry
 technology       1173
 retail            925
 healthcare        840
 finance           720
 education         639
 manufacturing     463
 NA                240
 Name: count, dtype: int64,
 0    technology
 Name: industry, dtype: str)

Question 2:

In [36]:
correlation_matrix = df[num_cols].corr()
correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


Split the data set

In [38]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values
del df_train["converted"]
del df_val["converted"]
del df_test["converted"]

Question 3:

In [40]:
from sklearn.metrics import mutual_info_score


def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)


mi = df_train[cat_cols].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

Question 4:

OHE

In [45]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

features = list(cat_cols) + list(num_cols)
features.remove("converted")
train_dict = df_train[features].to_dict(orient="records")
X_train = dv.fit_transform(train_dict)

val_dict = df_val[features].to_dict(orient="records")
X_val = dv.transform(val_dict)

Training model

In [62]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = y_pred >= 0.5
(y_val == converted_decision).mean().round(2)

np.float64(0.64)

Question 5:

In [66]:
model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = y_pred >= 0.5
original_accuracy = (y_val == converted_decision).mean()

for col in features:
    new_features = features.copy()
    new_features.remove(col)
    train_dict = df_train[new_features].to_dict(orient="records")
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[new_features].to_dict(orient="records")
    X_val = dv.transform(val_dict)
    model = LogisticRegression(
        solver="liblinear", C=1.0, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = y_pred >= 0.5
    print(col, original_accuracy - (y_val == converted_decision).mean())

lead_source 0.0020000000000000018
industry -0.0010000000000000009
employment_status -0.0010000000000000009
location -0.0010000000000000009
annual_income -0.07999999999999996
number_of_courses_viewed 0.0010000000000000009
interaction_count 0.04300000000000004
lead_score 0.0


Question 6:

In [68]:
from sklearn.linear_model import LogisticRegression

c = [0.000001, 0.00001, 0.0001, 0.001]

for value in c:
    model = LogisticRegression(
        solver="liblinear", C=value, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = y_pred >= 0.5
    print(value, (y_val == converted_decision).mean())

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.644
